# Professional Practice: Clean Code, CI/CD, Secrets, Security and Packaging

Notebook for **Chapter 13** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 Clean Code and Refactoring

In [ ]:
# BEFORE
def f(d, t):
    r = []
    for x in d:
        if x[2] == "DELIVERED":
            if x[1] > t:
                r.append(x[0])
    return r

rows = [("A101", 45, "DELIVERED"), ("A102", 12, "DELIVERED"), ("A103", 230, "LOST"), ("A104", 150, "DELIVERED")]
print(f(rows, 60))

# AFTER
from dataclasses import dataclass

@dataclass(frozen=True)
class Order:
    order_id: str
    delivery_minutes: int
    status: str

def slow_delivered_orders(orders, threshold_minutes):
    """Return ids of delivered orders slower than threshold_minutes."""
    return [o.order_id for o in orders if o.status == "DELIVERED" and o.delivery_minutes > threshold_minutes]

orders = [Order(*r) for r in rows]
print(slow_delivered_orders(orders, 60))
print(slow_delivered_orders(orders, 10))
print(f(rows, 10) == slow_delivered_orders(orders, 10))

['A104']
['A104']
['A101', 'A102', 'A104']
True


In [ ]:
import math

# BEFORE: the formula is copy-pasted three times
def p50(xs): xs = sorted(xs); return xs[max(0, math.ceil(len(xs) * 50 / 100) - 1)]
def p90(xs): xs = sorted(xs); return xs[max(0, math.ceil(len(xs) * 90 / 100) - 1)]
def p95(xs): xs = sorted(xs); return xs[max(0, math.ceil(len(xs) * 95 / 100) - 1)]

# AFTER: one function, one place to fix bugs
def percentile(xs, p):
    """Nearest-rank percentile, p in 1..100."""
    xs = sorted(xs)
    return xs[max(0, math.ceil(len(xs) * p / 100) - 1)]

data = [80, 95, 110, 120, 300]
print(p50(data), p90(data), p95(data))
print([percentile(data, p) for p in (50, 90, 95)])

## 13.2 Architecture: Layers, Monoliths, Microservices, Events and Queues

In [ ]:
from unittest.mock import Mock

class OrderRepository:                      # a real one would use a database
    def __init__(self): self.rows = {}
    def save(self, oid, total): self.rows[oid] = total
    def get(self, oid): return self.rows.get(oid)

class FakeRepo(OrderRepository): pass       # in-memory fake: real behaviour, no DB

class OrderService:
    def __init__(self, repo, notifier):     # dependencies injected
        self.repo, self.notifier = repo, notifier
    def place(self, oid, prices):
        total = sum(prices)
        self.repo.save(oid, total)
        self.notifier.send(f"order {oid}: {total}")
        return total

notifier = Mock()
svc = OrderService(FakeRepo(), notifier)
print(svc.place(1, [100, 250, 50]))
print(svc.repo.get(1))
notifier.send.assert_called_once_with("order 1: 400")
print(notifier.send.call_args)

400
400
call('order 1: 400')


In [ ]:
import queue, threading

q = queue.Queue()
results = []
def worker():
    while True:
        job = q.get()
        if job is None:
            break
        results.append(job * job)
        q.task_done()
t = threading.Thread(target=worker); t.start()
for n in range(1, 6): q.put(n)
q.put(None); t.join()
print(results)

handlers = {}
def on(evt):
    def deco(fn): handlers.setdefault(evt, []).append(fn); return fn
    return deco
def emit(evt, **data):
    for h in handlers.get(evt, []): h(**data)
@on("order_placed")
def send_email(order_id): print("email for", order_id)
@on("order_placed")
def update_stock(order_id): print("stock updated for", order_id)
emit("order_placed", order_id=42)

[1, 4, 9, 16, 25]
email for 42
stock updated for 42


## 13.3 Agile, Estimation, Technical Debt, Code Review and Documentation

In [ ]:
import math

def percentile(values, p):
    """Nearest-rank percentile.

    >>> percentile([80, 95, 110, 120, 300], 90)
    300
    >>> percentile([1, 2, 3, 4], 50)
    2
    """
    xs = sorted(values)
    k = max(0, math.ceil(len(xs) * p / 100) - 1)
    return xs[k]

## 13.4 CI/CD Basics

In [ ]:
import subprocess, sys

def stage(name, code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    print(f"{name:6} -> {'PASS' if r.returncode == 0 else 'FAIL'}")
    return r.returncode == 0

stages = [
    ("lint", "pass"),
    ("test", "assert 2 + 2 == 5"),
    ("deploy", "print('shipping')"),
]
for name, code in stages:
    if not stage(name, code):
        print("pipeline stopped at", name)
        break
else:
    print("pipeline passed")

lint   -> PASS
test   -> FAIL
pipeline stopped at test


## 13.5 Configuration and Secrets Handling

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import os
from dataclasses import dataclass

os.environ["PIPELINE_ENV"] = "prod"      # in real life the shell or CI sets this

@dataclass(frozen=True)
class Settings:
    env: str
    batch_size: int = 500
    @classmethod
    def from_env(cls):
        return cls(env=os.environ["PIPELINE_ENV"],
                   batch_size=int(os.environ.get("BATCH_SIZE", "500")))

print(Settings.from_env())

try:
    os.environ["MISSING_REQUIRED"]
except KeyError as e:
    print("KeyError", e)
print(os.environ.get("MISSING", "default"))

Settings(env='prod', batch_size=500)
KeyError 'MISSING_REQUIRED'
default


In [ ]:
import re

def load_dotenv(text):
    out = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line: continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip().strip('"')
    return out
print(load_dotenv('# demo\nAPI_KEY="fake-key-123"\nDEBUG=1\n'))

def mask_email(s): return re.sub(r"(\w)[\w.]*@", r"\1***@", s)
def mask_phone(s): return re.sub(r"\b\d{6}(\d{4})\b", r"XXXXXX\1", s)
print(mask_email("asha.k@example.com"), mask_phone("9876543210"))

pat = re.compile(r"(?i)(api[_-]?key|secret|token)\s*=\s*['\"][A-Za-z0-9_\-]{8,}['\"]")
code = 'x = 1\nAPI_KEY = "fake-key-123"\nname = "abc"\n'
for i, l in enumerate(code.splitlines(), 1):
    if pat.search(l): print("line", i, "looks like a secret")

{'API_KEY': 'fake-key-123', 'DEBUG': '1'}
a***@example.com XXXXXX3210
line 2 looks like a secret


In [ ]:
import logging, re
class Redact(logging.Filter):
    def filter(self, record):
        record.msg = re.sub(r"(token=)\w+", r"\1***", str(record.msg)); return True
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("app"); log.addFilter(Redact())
log.info("calling api token=abc123xyz")

INFO calling api token=***


## 13.6 SQL Injection, Password Hashing and Safe Basics

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT, secret TEXT);
INSERT INTO users(name, secret) VALUES ('asha','pw-asha'),('ravi','pw-ravi');
""")

def vulnerable(name):
    q = f"SELECT id, name FROM users WHERE name = '{name}'"
    print("SQL:", q)
    return con.execute(q).fetchall()

def safe(name):
    q = "SELECT id, name FROM users WHERE name = ?"
    return con.execute(q, (name,)).fetchall()

print(vulnerable("asha"))
attack = "x' OR '1'='1"
print(vulnerable(attack))
print(safe("asha"))
print(safe(attack))
attack2 = "x' UNION SELECT id, secret FROM users --"
print(vulnerable(attack2))
print(safe(attack2))

SQL: SELECT id, name FROM users WHERE name = 'asha'
[(1, 'asha')]
SQL: SELECT id, name FROM users WHERE name = 'x' OR '1'='1'
[(1, 'asha'), (2, 'ravi')]
[(1, 'asha')]
[]
SQL: SELECT id, name FROM users WHERE name = 'x' UNION SELECT id, secret FROM users --'
[(1, 'pw-asha'), (2, 'pw-ravi')]
[]


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT)")
con.execute("INSERT INTO users(name) VALUES (?)", ("O'Brien",))
name = "O'Brien"
try:
    con.execute(f"SELECT * FROM users WHERE name = '{name}'")
except sqlite3.Error as e:
    print(type(e).__name__, e)
print(con.execute("SELECT * FROM users WHERE name = ?", (name,)).fetchall())

OperationalError near "Brien": syntax error
[(1, "O'Brien")]


In [ ]:
import hashlib, os, hmac, uuid

def hash_pw(pw, salt=None):
    salt = salt or os.urandom(16)
    dk = hashlib.scrypt(pw.encode(), salt=salt, n=2**14, r=8, p=1)
    return salt, dk

def check(pw, salt, dk):
    return hmac.compare_digest(hash_pw(pw, salt)[1], dk)

salt, dk = hash_pw("correct horse")
print(len(salt), len(dk))
print(check("correct horse", salt, dk), check("wrong", salt, dk))
s2, d2 = hash_pw("correct horse")
print("same password, different salt, same hash?", dk == d2)

print(hashlib.sha256(b"hello").hexdigest())
print(hashlib.sha256(b"hello!").hexdigest())
print(hashlib.md5(b"hello").hexdigest(), "(md5: fine for checksums, never for security)")
u = uuid.uuid4(); print(len(str(u)), u.version)
print(uuid.uuid5(uuid.NAMESPACE_DNS, "example.com"))
print(uuid.uuid5(uuid.NAMESPACE_DNS, "example.com"))

16 64
True False
same password, different salt, same hash? False
2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
ce06092fb948d9ffac7d1a376e404b26b7575bcc11ee05a4615fef4fec3a308b
5d41402abc4b2a76b9719d911017c592 (md5: fine for checksums, never for security)
36 4
cfbff0d1-9375-5685-968c-48ce8b15ae17
cfbff0d1-9375-5685-968c-48ce8b15ae17


## 13.7 Packaging, Dependencies and Supply-Chain Safety

In [ ]:
import tomllib, zipfile, sys, pathlib, doctest
root = pathlib.Path("pk")
cfg = tomllib.loads((root / "pyproject.toml").read_text())
print(cfg["project"]["name"], cfg["project"]["version"], cfg["project"]["scripts"])
whl = root / "mini_stats-0.1.0-demo.zip"
with zipfile.ZipFile(whl, "w") as z:
    for f in (root / "src").rglob("*.py"):
        z.write(f, f.relative_to(root / "src"))
print(zipfile.ZipFile(whl).namelist())
sys.path.insert(0, str(whl))
import mini_stats
print(mini_stats.__version__, mini_stats.percentile([5, 1, 9, 3], 75))
import mini_stats.core as c
print(doctest.testmod(c))

mini-stats 0.1.0 {'mini-stats': 'mini_stats.cli:main'}
['mini_stats/__init__.py', 'mini_stats/core.py']
0.1.0 5
TestResults(failed=0, attempted=2)


## 13.8 Production Habits: Structured Logs, Secrets, Safe Subprocesses and a Slim Container

Chapter 6 taught logging for a script you run yourself. A service run by a team at 3 a.m. needs logs a machine can search, secrets nobody can guess, and a container nobody can easily break out of. This section adds those habits. The figure below is the map to keep in mind: requests flow down through layers, and every layer should log with the same request id.

In [ ]:
import contextvars, json, logging, sys

request_id = contextvars.ContextVar("request_id", default="-")

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {"level": record.levelname, "logger": record.name,
                   "msg": record.getMessage(), "request_id": request_id.get()}
        payload.update(getattr(record, "ctx", {}))
        return json.dumps(payload, sort_keys=True)

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JsonFormatter())
log = logging.getLogger("shop")
log.addHandler(handler)
log.setLevel(logging.INFO)
log.propagate = False

def charge(amount):
    log.info("charging card", extra={"ctx": {"amount": amount}})

def handle(req_id, amount):
    token = request_id.set(req_id)
    try:
        log.info("request start")
        charge(amount)
    finally:
        request_id.reset(token)

handle("req-a1", 20)
handle("req-b2", 35)
log.info("outside any request")

{"level": "INFO", "logger": "shop", "msg": "request start", "request_id": "req-a1"}
{"amount": 20, "level": "INFO", "logger": "shop", "msg": "charging card", "request_id": "req-a1"}
{"level": "INFO", "logger": "shop", "msg": "request start", "request_id": "req-b2"}
{"amount": 35, "level": "INFO", "logger": "shop", "msg": "charging card", "request_id": "req-b2"}
{"level": "INFO", "logger": "shop", "msg": "outside any request", "request_id": "-"}


In [ ]:
import logging, logging.config, io

stream = io.StringIO()
CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {"plain": {"format": "%(levelname)s %(name)s: %(message)s"}},
    "handlers": {"mem": {"class": "logging.StreamHandler", "stream": stream, "formatter": "plain"}},
    "loggers": {"shop": {"level": "INFO", "handlers": ["mem"], "propagate": False},
                "shop.db": {"level": "WARNING"}},
}
logging.config.dictConfig(CONFIG)
logging.getLogger("shop").info("started")
logging.getLogger("shop.db").info("hidden: db logger is WARNING")
logging.getLogger("shop.db").warning("slow query")
print(stream.getvalue(), end="")

INFO shop: started
WARNING shop.db: slow query


In [ ]:
import secrets, string
token = secrets.token_urlsafe(16)
print(len(token) >= 21, set(token) <= set(string.ascii_letters + string.digits + "-_"))
print(secrets.compare_digest("abc", "abc"), secrets.compare_digest("abc", "abd"))
alphabet = string.ascii_lowercase + string.digits
code = "".join(secrets.choice(alphabet) for _ in range(8))
print(len(code))

True True
True False
8


In [ ]:
import subprocess, sys
filename = "notes.txt; echo INJECTED"
# DANGEROUS: the shell reads the whole string as commands
bad = subprocess.run(f"echo reading {filename}", shell=True, capture_output=True, text=True)
print(bad.stdout.strip().splitlines())
# SAFE: a list of arguments, no shell
good = subprocess.run(["echo", "reading", filename], capture_output=True, text=True)
print(good.stdout.strip().splitlines())

['reading notes.txt', 'INJECTED']
['reading notes.txt; echo INJECTED']


In [ ]:
import pickle
class Evil:
    def __reduce__(self):
        return (print, ("code ran during pickle.loads",))
blob = pickle.dumps(Evil())
pickle.loads(blob)

code ran during pickle.loads


## Chapter recap

## Practice set

Problem 13.P.1 (easy) What is wrong with the name `def f(d, t):`, and what would you change?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch13.html